In [1]:
#直接使用GPT2模型来计算奖励,输入是一个文本序列，输出是该序列的奖励值。gpt+线性层
import torch
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer



class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        # llm最后输出的隐藏层的维度
        self.hidden_size = config.hidden_size
        # 线性层用来对llm最后输出的隐藏层给奖励
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        # 使用正态分布初始化权重
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        # 将偏置初始化为0
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        # 给出奖励
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_head = RewardHead(self.llm.config)

    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1] # 获取最后一层的隐藏状态，而不是输出的logits
        # 给出奖励
        reward = self.reward_head(last_hidden_state).squeeze(-1)
        # sigmoid用来将奖励搞到(-1,1)范围内
        return torch.sigmoid(reward)

model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
model = GPT2RewardHead(model_path)


D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.regi

In [2]:
from functools import partial
#加载tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer


# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch, tokenizer, REWARD_TOKEN_ID):
    # 分词
    contexts = tokenizer(batch['context'])
    batch_size = len(contexts['input_ids'])  # 批次大小

    # 初始化得分和得分位置（长度 = batch_size）
    contexts["score"] = [0.0] * batch_size
    contexts["score_index"] = [0] * batch_size

    for i in range(batch_size):
        # 在每条文本末尾添加 reward token（eos token）
        contexts['input_ids'][i].append(REWARD_TOKEN_ID)
        contexts['attention_mask'][i].append(1)

        # 评分：label 是 1（正向）或 0（负向）
        contexts['score'][i] = float(batch['label'][i])

        # 记录 reward token 的索引（最后一个位置）
        contexts['score_index'][i] = len(contexts['input_ids'][i]) - 1

    return contexts


#清理数据长度国小的
def filter_short_samples(batch, min_length=10):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)

def load_data(data_path,tokenizer):
    dataset = load_dataset('json', data_files={
        'train': data_path[0],
        'valid': data_path[1],
        'test': data_path[2]
    })
    # 查看结构
    print(dataset['train'][0])
    # 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

    map_kwargs = {
        'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
        'batch_size': 512,  # 每批 512 个样本
        'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
    }
    REWARD_TOKEN_ID = tokenizer.eos_token_id
    print(REWARD_TOKEN_ID)
    # 在 map 时绑定 tokenizer
    tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer,REWARD_TOKEN_ID = REWARD_TOKEN_ID)

    # 对训练集和验证集应用 map
    tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
    tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

    print(tokenized_dataset_train[0])
    print(tokenized_dataset_val[0])


    # 对训练集和验证集应用过滤函数
    filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
    filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
    print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
    print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
    print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
    print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

    # 放入torch
    import torch
    filtered_dataset_train.set_format(type='torch')
    filtered_dataset_val.set_format(type='torch')
    print(filtered_dataset_train[0])
    print(filtered_dataset_val[0])


    return filtered_dataset_train, filtered_dataset_val, dataset["test"]

data_path = ["../SFT微调大模型/data_raw/train.jsonl","../SFT微调大模型/data_raw/valid.jsonl","../SFT微调大模型/data_raw/test.jsonl"]
tokenizer = AutoTokenizer.from_pretrained(model_path)
filtered_dataset_train, filtered_dataset_val, dataset_test = load_data(data_path=data_path,tokenizer=tokenizer)

{'idx': 0, 'context': '死囚 爱 刽子手 女贼 爱 衙役 我们 爱 你们 难道 还有 别的 选择 没想到 胡军 除了 蓝宇 还有 东宫 西宫 我 个 去 阿兰 这样 真 他 nia 恶心 爱个 P 分明 只是 欲', 'label': 1}
50256
{'input_ids': [29826, 119, 32368, 248, 13328, 230, 109, 10263, 230, 121, 36310, 33699, 233, 10263, 98, 111, 164, 112, 120, 13328, 230, 109, 5525, 94, 247, 37605, 117, 10545, 230, 239, 20015, 105, 13328, 230, 109, 220, 19526, 254, 20015, 105, 16268, 248, 122, 34402, 241, 5525, 123, 246, 17312, 231, 10263, 230, 104, 21410, 16268, 222, 231, 162, 233, 102, 10545, 110, 94, 46349, 111, 26344, 108, 5525, 225, 94, 37863, 249, 16268, 247, 97, 12859, 228, 5525, 241, 251, 22522, 229, 5525, 123, 246, 17312, 231, 220, 10310, 250, 22522, 104, 5525, 98, 123, 22522, 104, 10545, 230, 239, 220, 10310, 103, 10263, 236, 119, 16268, 246, 123, 17739, 108, 5525, 123, 247, 43718, 115, 13328, 250, 253, 220, 20015, 244, 299, 544, 10545, 223, 114, 33232, 225, 13328, 230, 109, 10310, 103, 350, 10263, 230, 228, 23626, 236, 10263, 237, 103, 42468, 10545, 105, 110, 50256], 'attent

In [3]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
# 将 eos token 作为 pad token
tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer)
dataloader_params = {
    'batch_size': 16,
    'shuffle': True,
    'collate_fn': data_collator
}
train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

batch = next(iter(train_dataloader))
print(batch.keys())

print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])
print(tokenizer.decode(batch['input_ids'][1]))
print(batch['attention_mask'][1].nonzero()[-1])

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


dict_keys(['input_ids', 'attention_mask', 'score', 'score_index'])
tensor([  164, 41678,   164,   226,   109, 36469,   100, 32368,    95, 28839,
          228, 13328,   119,   241,   161,   109,   222,   220, 19526,   228,
        42468, 10263,   122,   230, 13328,   250,   253, 22522,   252, 10263,
          120,   118,   164,   108,   225,   220, 27670,   254,   163,   119,
          253, 13328,   242,   253,   162,   112,   119,   220, 20015,   115,
          161,   222,   120, 10263,   237,   107,   164,   112,   113, 10263,
          106,   231,   163,   101,   111,   220, 38834, 42468,   220, 19526,
          254, 10545,   225,   111, 17358,   223, 10545,   225,   111, 17358,
          223,  5525,   225,   121,  5525,    99,   223, 13328,   250,   233,
        12490, 13328,   105,   105, 31660, 32849,   101,   220, 20015,   236,
         1995,    64, 13328,    95,   253,  2041, 16268,   229,   234,   220,
        20046,   242,   162,   110,   119, 10263, 26534,   165,   110,   22

In [4]:
outputs = model(batch['input_ids'], batch['attention_mask'])
print(outputs.shape)

torch.Size([16, 146])


In [12]:
#论文里面是只训练了一个epoch，奖励模型的训练通常需要较少的epoch，因为它们通常是用来评估生成模型的输出，而不是直接生成文本。过多的训练可能会导致奖励模型过拟合，从而无法有效地评估生成模型的输出。
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
# 二分类交叉熵损失
criterion = nn.BCELoss()
num_epochs = 2


In [13]:
#评估
def validate():
    model.eval()
    total_loss = 0
    for i, batch in enumerate(val_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        with torch.no_grad():
            # 对输出进行评分
            scores = model(**model_inputs)
            # 批次中每条数据的索引
            batch_indices = torch.arange(scores.shape[0])
            # 根据索引拿出评分，也就是reward token的评分
            score = scores[batch_indices, inputs['score_index']]
            # 目标评分，0 或者 1 。
            target = inputs['score']  #需要注意的是这里的评分都是大于0的，但是可以通过调整reward token的位置来让模型学会给负向样本打分更低，给正向样本打分更高。
            # 计算误差
            loss = criterion(score, target)
        total_loss += loss.item()
    print('validation loss:', total_loss / len(val_dataloader))

#训练
model.to(device)

validate()
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
        loss = criterion(score, target)
        #清空梯度 ⟶ 反向传播计算梯度 ⟶ 更新参数
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())
    validate()

validation loss: 0.8320396086748909
0.7306556105613708
0.8102135062217712
0.6967700123786926
0.6580839157104492
0.7165414094924927
0.7145531177520752
0.6705037355422974
0.7983354330062866
0.6927950382232666
0.704494833946228
0.6536743640899658
0.7227237224578857
0.6988030672073364
0.6934079527854919
0.6495569944381714
0.7110865712165833
0.7384520769119263
0.6860598921775818
0.6606141328811646
0.7309836149215698
0.6891663074493408
0.6961318254470825
0.6858969926834106
0.6754436492919922
0.7168716192245483
0.7155752182006836
0.6708059310913086
0.6971187591552734
0.7343946099281311
0.6742966771125793
0.6890875101089478
0.7141640782356262
0.6824687719345093
0.6879338622093201
0.7062153816223145
0.7095378041267395
0.6743899583816528
0.7189861536026001
0.7245633602142334
0.7436447143554688
0.6840936541557312
0.6987506151199341
0.6976059675216675
0.6652593612670898
0.6861745119094849
0.732850968837738
0.6715908050537109
0.6840001344680786
0.6718037724494934
0.6644877195358276
0.68624830245971

In [14]:
torch.save(model.state_dict(), 'models/reward_model.pt')

In [15]:
#评估
validate()

validation loss: 0.6813250610641405
